# Talking to the MCP Server Directly

No LLM and no agent: just an MCP client that
1. lists the tools the server offers, and
2. calls the `health` tool.

```
┌──────────────────────────┐   list_tools()     ┌──────────────────────┐
│ MCP client (notebook)    ├───────────────────►│ MCP server           │
│                          │   call_tool()      │ (Databricks App)     │
│                          │◄───────────────────┤                      │
└──────────────────────────┘   results          └──────────────────────┘
```

In [ ]:
%pip install -q "fastmcp>=2.10.0" "databricks-sdk>=0.50.0"

## Configuration and authentication

Same service principal setup (OAuth M2M) as in the other notebooks.

In [ ]:
PROFILE = "resa-prod"                # Databricks CLI profile (only used with LOCAL_MCP)
APP_NAME = "mcp-server-template"    # Databricks App hosting the MCP server
LOCAL_MCP = False                   # True -> use `make run` server on localhost:8000

In [ ]:
from databricks.sdk import WorkspaceClient

if LOCAL_MCP:
    w = WorkspaceClient(profile=PROFILE)
    mcp_url, mcp_headers = "http://localhost:8000/mcp", {}
else:
    w = WorkspaceClient(
        host=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get(),  # current workspace
        client_id=dbutils.secrets.get("service-principals", "mcp-server-template-user-client-id"),
        client_secret=dbutils.secrets.get("service-principals", "mcp-server-template-user-client-secret"),
        auth_type="oauth-m2m",
    )
    mcp_url = w.apps.get(APP_NAME).url.rstrip("/") + "/mcp"
    mcp_headers = w.config.authenticate()  # {"Authorization": "Bearer <oauth-token>"}

print("MCP URL:", mcp_url)

## 1. List the tools

In [ ]:
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport

def mcp_client() -> Client:
    return Client(StreamableHttpTransport(mcp_url, headers=mcp_headers))

async with mcp_client() as mcp:
    tools = await mcp.list_tools()

for tool in tools:
    print(f"- {tool.name}: {(tool.description or '').strip().splitlines()[0]}")

## 2. Call the `health` tool

In [ ]:
async with mcp_client() as mcp:
    result = await mcp.call_tool("health", {})

print(result.content[0].text)